Setup

In [1]:
import time
import json
import csv

from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap, PassManager
from pathlib import Path
from typing import Dict, List, Iterable
from util import EAGLE_COUPLING, sabre, count_swaps 
from multilevel_sabre import MultiLevelSabre
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes import VF2Layout

random_seed = 1

Coupling map

In [2]:
from util import EAGLE_COUPLING

coupling_map = CouplingMap(couplinglist=EAGLE_COUPLING)
coupling_map.make_symmetric()

Select Circuits

In [3]:
directory = "circuits_no_swaps/"
result_path = Path("results/comparison_no_swaps.csv")
circuit_files = []
for file in Path(directory).glob("*.qasm"):
    circuit_files.append(file)
print(f"Found {len(circuit_files)} circuit files.")    

Found 7 circuit files.


Pass manager setup

In [ ]:
#basis_gates=["rz", "sx", "x", "cx"]
pm = generate_preset_pass_manager(optimization_level=3, coupling_map=coupling_map, seed_transpiler=random_seed)
vf2_layout = VF2Layout(coupling_map=coupling_map, seed=1, time_limit=10)
pm.layout.replace(1, vf2_layout)


Define callback function to get info of the passes during transpilation, in particular the VF2Layout and PostVF2Layout passes. 

In [5]:
passes = []
def callback_func(**kwargs):
    dag_name = kwargs['dag'].name
    t_pass_name = kwargs['pass_'].name()
    t_property = kwargs['property_set']

    # Record the pass name with its corresponding property set
    passes.append((dag_name, t_pass_name, t_property))


Run circuit

In [6]:
passes = []

for qasm_file in circuit_files:
    qc = QuantumCircuit.from_qasm_file(qasm_file)
    qc.name = qasm_file.stem
    qc = qc.decompose(reps=3)
    qc_tr = pm.run(qc, callback=callback_func, output_name=qc.name)
    
    swaps = count_swaps(qc_tr)
    depth_2q = qc_tr.depth(lambda x: x.operation.num_qubits == 2)
    print(f"{qc.name:<20} {swaps:>6} swaps, {depth_2q:>6} 2q depth")
    count_ops = qc_tr.count_ops()
    print(f"  Ops: {count_ops}")


swap_test_n83           153 swaps,    319 2q depth
  Ops: OrderedDict({'cx': 246, 'p': 165, 'swap': 153, 'unitary': 41, 'u2': 41, 'u': 2, 'measure': 1})
knn_67                  124 swaps,    264 2q depth
  Ops: OrderedDict({'cx': 198, 'p': 133, 'swap': 124, 'unitary': 33, 'u2': 33, 'u': 2, 'measure': 1})
dnn_n51                 106 swaps,    191 2q depth
  Ops: OrderedDict({'cx': 150, 'swap': 106, 'p': 101, 'unitary': 73, 'measure': 51, 'u2': 25, 'u': 2, 'barrier': 1})
ghz_n78                   0 swaps,     77 2q depth
  Ops: OrderedDict({'measure': 78, 'cx': 77, 'u': 1, 'barrier': 1})
ising_n98                 0 swaps,      4 2q depth
  Ops: OrderedDict({'cx': 194, 'u': 145, 'u2': 98, 'measure': 98, 'u1': 48, 'barrier': 1})
multiplier_n45         1550 swaps,   2078 2q depth
  Ops: OrderedDict({'u': 2335, 'cx': 2286, 'swap': 1550, 'u1': 244, 'u2': 170, 'measure': 9})
cat_n65                   0 swaps,     64 2q depth
  Ops: OrderedDict({'measure': 65, 'cx': 64, 'u': 1, 'barrier': 1})


In [7]:

curr_dag = None
for dag_name, pass_name, property_set in passes:
    if curr_dag != dag_name:
        curr_dag = dag_name
        print(f"Circuit: {dag_name}")
    if pass_name == "VF2Layout":
        print(f"  Pass: {pass_name}")
        print(f"    Stop reason: {property_set['VF2Layout_stop_reason']}")
    elif pass_name == "VF2PostLayout":
        print(f"  Pass: {pass_name}")
        print(f"    Stop reason: {property_set['VF2PostLayout_stop_reason']}")

Circuit: swap_test_n83
  Pass: VF2Layout
    Stop reason: VF2LayoutStopReason.NO_SOLUTION_FOUND
  Pass: VF2PostLayout
    Stop reason: VF2PostLayoutStopReason.NO_SOLUTION_FOUND
  Pass: VF2PostLayout
    Stop reason: VF2PostLayoutStopReason.NO_SOLUTION_FOUND
Circuit: knn_67
  Pass: VF2Layout
    Stop reason: VF2LayoutStopReason.NO_SOLUTION_FOUND
  Pass: VF2PostLayout
    Stop reason: VF2PostLayoutStopReason.NO_SOLUTION_FOUND
  Pass: VF2PostLayout
    Stop reason: VF2PostLayoutStopReason.NO_SOLUTION_FOUND
Circuit: dnn_n51
  Pass: VF2Layout
    Stop reason: VF2LayoutStopReason.NO_SOLUTION_FOUND
  Pass: VF2PostLayout
    Stop reason: VF2PostLayoutStopReason.NO_SOLUTION_FOUND
  Pass: VF2PostLayout
    Stop reason: VF2PostLayoutStopReason.NO_SOLUTION_FOUND
Circuit: ghz_n78
  Pass: VF2Layout
    Stop reason: VF2LayoutStopReason.SOLUTION_FOUND
  Pass: VF2PostLayout
    Stop reason: VF2PostLayoutStopReason.NO_SOLUTION_FOUND
Circuit: ising_n98
  Pass: VF2Layout
    Stop reason: VF2LayoutStopReas